In [1]:
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import random

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

In [3]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D,
    BatchNormalization, Dropout,
    Dense, GlobalAveragePooling2D
)

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [23]:
# Fix Randomness (IMPORTANT)

In [5]:
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

In [22]:
# Plant Classes

In [7]:
plants = [
'Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy',
'Potato___Early_blight', 'Potato___healthy',
'Potato___Late_blight', 'Tomato_Bacterial_spot',
'Tomato_Early_blight', 'Tomato_healthy',
'Tomato_Late_blight', 'Tomato_Leaf_Mold',
'Tomato_Septoria_leaf_spot',
'Tomato_Spider_mites_Two_spotted_spider_mite',
'Tomato__Target_Spot',
'Tomato__Tomato_mosaic_virus',
'Tomato__Tomato_YellowLeaf__Curl_Virus'
]

In [24]:
# Load Dataset

In [9]:
print("Loading dataset...")

X = np.load("Model/myimg_data.txt.npy")
Y = np.load("Model/myimg_label.txt.npy")

# Normalize
X = X.astype("float32") / 255.0

# One-hot labels
Y = to_categorical(Y, num_classes=15)

print("Dataset Shape:", X.shape)

Loading dataset...
Dataset Shape: (20638, 64, 64, 3)


In [25]:
# Train / Test Split (Fixed)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

In [26]:
# Build Optimized CNN Model

In [13]:
print("Building model...")

inputs = Input(shape=(64,64,3))

x = Conv2D(32,(3,3),padding='same',activation='relu')(inputs)
x = BatchNormalization()(x)
x = MaxPooling2D((2,2))(x)

x = Conv2D(64,(3,3),padding='same',activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2,2))(x)

x = Conv2D(128,(3,3),padding='same',activation='relu', name="last_conv")(x)
x = BatchNormalization()(x)
x = MaxPooling2D((2,2))(x)

# Replace Flatten → GAP
x = GlobalAveragePooling2D()(x)

x = Dense(128,activation='relu')(x)
x = Dropout(0.5)(x)

outputs = Dense(15,activation='softmax')(x)

model = Model(inputs,outputs)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Building model...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ last_conv (Conv2D)              │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │         1,935 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 112,591 (439.81 KB)

 Trainable params: 112,143 (438.06 KB)

 Non-trainable params: 448 (1.75 KB)

In [27]:
# Training

In [15]:
print("\nTraining model...")

early_stop = EarlyStopping(
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3
)

history = model.fit(
    X_train,
    y_train,
    epochs=25,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


Training model...
Epoch 1/25
413/413 ━━━━━━━━━━━━━━━━━━━━ 46s 104ms/step - accuracy: 0.6036 - loss: 1.2393 - val_accuracy: 0.2526 - val_loss: 7.9932 - learning_rate: 0.0010
Epoch 2/25
413/413 ━━━━━━━━━━━━━━━━━━━━ 80s 99ms/step - accuracy: 0.7650 - loss: 0.7100 - val_accuracy: 0.5815 - val_loss: 1.3564 - learning_rate: 0.0010
Epoch 3/25
413/413 ━━━━━━━━━━━━━━━━━━━━ 43s 103ms/step - accuracy: 0.8364 - loss: 0.5027 - val_accuracy: 0.7002 - val_loss: 0.9063 - learning_rate: 0.0010
Epoch 4/25
413/413 ━━━━━━━━━━━━━━━━━━━━ 42s 102ms/step - accuracy: 0.8792 - loss: 0.3732 - val_accuracy: 0.3537 - val_loss: 2.8920 - learning_rate: 0.0010
Epoch 5/25
413/413 ━━━━━━━━━━━━━━━━━━━━ 42s 103ms/step - accuracy: 0.9009 - loss: 0.3050 - val_accuracy: 0.7941 - val_loss: 0.6424 - learning_rate: 0.0010
Epoch 6/25
413/413 ━━━━━━━━━━━━━━━━━━━━ 42s 102ms/step - accuracy: 0.9224 - loss: 0.2423 - val_accuracy: 0.6705 - val_loss: 1.2140 - learning_rate: 0.0010
Epoch 7/25
413/413 ━━━━━━━━━━━━━━━━━━━━ 55s 133ms/st

In [28]:
# Save Model

In [21]:
model.save("model/plant_disease_model.h5")
print("\nModel saved!")


Model saved!


In [29]:
# Evaluation

In [20]:
print("\nEvaluating model...")

loss, accuracy = model.evaluate(X_test, y_test)

print("\nTest Accuracy:", accuracy)

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print("\nPrecision:", precision_score(y_true,y_pred_classes,average='weighted'))
print("Recall:", recall_score(y_true,y_pred_classes,average='weighted'))
print("F1 Score:", f1_score(y_true,y_pred_classes,average='weighted'))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true,y_pred_classes))

print("\nClassification Report:\n")
print(classification_report(y_true,y_pred_classes,target_names=plants))


Evaluating model...
129/129 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9724 - loss: 0.0837

Test Accuracy: 0.9723837375640869
129/129 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step

Precision: 0.9726495893559708
Recall: 0.9723837209302325
F1 Score: 0.9723183445039746

Confusion Matrix:

[[171   3   0   0   0   1   0   0   0   0   4   0   0   0   0]
 [  2 268   0   0   0   0   0   0   1   0   0   0   0   0   0]
 [  0   0 215   0   1   0   0   0   0   0   0   0   0   0   0]
 [  0   0   0  32   2   0   0   0   1   0   0   3   1   0   0]
 [  0   0   0   0 207   0   1   0   4   0   0   0   1   0   0]
 [  0   0   0   0   1 419   0   0   0   0   1   0   0   0   2]
 [  1   0   1   0   0   6 185   0   3   0   1   1   1   1   0]
 [  0   0   0   0   0   0   0 324   2   0   0   0   1   0   0]
 [  1   1   0   0   5   1  10   0 370   1   0   0   0   0   1]
 [  0   0   0   0   0   0   1   0   0 167   2   1   0   0   0]
 [  1   0   2   0   0   0   1   0   0   2 315   0   1   0   0]
 [  0   0   0   0   0   0 